# Brain Age Prediction Training Notebook per Kaggle
Questo notebook esegue un addestramento con la seguente architettura di valutazione:
1. **Hold-out Test Set**: Estrae il 10% del dataset in modo STRATIFICATO rispetto all'età.
2. **5-Fold Cross Validation Stratificata**: Sul restante 90% del dataset esegue un K-Fold mantenendo le proporzioni d'età:
   - Ottimizzatore SGD (LR=0.01, Weight Decay=0.001).
   - KL Divergence Loss convertendo le classi in gaussiane.
   - Data Augmentation: 50% shift random e 50% flip sagittale.
3. **Valutazione Finale**: Esegue il modello migliore sul 10% test set e ne analizza visivamente gli errori.
4. **Salvataggio Modello**: Salva i pesi (.pth) del modello migliore nella cartella di lavoro.

In [ ]:
!rm -rf SFCN
!git clone https://github.com/PietroSchgor/SFCN.git

import sys
sys.path.append('/kaggle/working/SFCN')

In [ ]:
import os
import json
import glob
import numpy as np
import nibabel as nib
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold, train_test_split

# Import dal tuo repository
from dp_model.model_files.sfcn import SFCN
from dp_model import dp_utils as dpu
from train import train_model

## 1. Dataset Custom (con Data Augmentation)

In [ ]:
class BrainAgeDataset(Dataset):
    def __init__(self, data_dir, is_train=True):
        self.data_dir = data_dir
        self.is_train = is_train
        self.subject_dirs = sorted(glob.glob(os.path.join(data_dir, "sub-*")))
        self.samples = []
        
        self.bin_range = [0, 70]
        self.bin_step = 1
        self.sigma = 1.0
        
        for subj_dir in self.subject_dirs:
            subj_id = os.path.basename(subj_dir)
            nii_path = os.path.join(subj_dir, f"{subj_id}_FLAIR_MNI152_1mm.nii")
            
            if not os.path.exists(nii_path):
                nii_path = nii_path + ".gz"
                if not os.path.exists(nii_path):
                    print(f"Saltato {subj_id}: NIfTI non trovato in {nii_path}")
                    continue
                    
            json_path = os.path.join(subj_dir, f"{subj_id}_participant_info.json")
            if not os.path.exists(json_path):
                print(f"Saltato {subj_id}: JSON non trovato in {json_path}")
                continue
                
            with open(json_path, 'r') as f:
                info = json.load(f)
                
            participant_info = info.get("participant_info", {})
            age_cat_val = participant_info.get("age_scan")
            
            if age_cat_val is None:
                print(f"Saltato {subj_id}: Chiave 'age_scan' assente in 'participant_info'.")
                continue
            
            try:
                age_cat = int(age_cat_val) - 1
                true_age = 3 + age_cat * 5
                
                y, _ = dpu.num2vect(true_age, self.bin_range, self.bin_step, self.sigma)
                
            except Exception as e:
                print(f"Saltato {subj_id}: Errore parsing age_scan = {age_cat_val}. Errore: {e}")
                continue
            
            self.samples.append({
                "nii_path": nii_path,
                "label_vect": y,
                "true_age": true_age
            })

        if len(self.samples) == 0:
            print(f"ATTENZIONE: Nessun campione trovato. Cerca le cartelle sub-* in: {data_dir}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        
        img = nib.load(sample['nii_path'])
        data = img.get_fdata(dtype=np.float32)
        
        mean_val = np.mean(data)
        if mean_val > 0:
            data = data / mean_val
            
        in_sp = data.shape
        out_sp = (160, 192, 160)
        
        if self.is_train:
            if np.random.rand() > 0.5:
                data = np.flip(data, axis=0).copy()
                
            dx = np.random.randint(-2, 3)
            dy = np.random.randint(-2, 3)
            dz = np.random.randint(-2, 3)
        else:
            dx, dy, dz = 0, 0, 0
            
        x_c = int((in_sp[0] - out_sp[0]) / 2) + dx
        y_c = int((in_sp[1] - out_sp[1]) / 2) + dy
        z_c = int((in_sp[2] - out_sp[2]) / 2) + dz
        
        data = data[x_c:x_c+out_sp[0], y_c:y_c+out_sp[1], z_c:z_c+out_sp[2]]
        data = np.expand_dims(data, axis=0)
        
        tensor_data = torch.from_numpy(data)
        label_vect = torch.tensor(sample['label_vect'], dtype=torch.float32)
        
        return tensor_data, label_vect, sample['true_age']

## 2. Distribuzione Età, Split Stratificati, Addestramento e Salvataggio

In [ ]:
KAGGLE_DATA_DIR = "/kaggle/input/datasets/elenaschgor/dataset-2-t1-flair/ds004199_final/"

full_train_dataset = BrainAgeDataset(KAGGLE_DATA_DIR, is_train=True)
full_val_dataset   = BrainAgeDataset(KAGGLE_DATA_DIR, is_train=False) 
dataset_size = len(full_train_dataset)

print(f"Trovati {dataset_size} campioni validi.\n")

if dataset_size > 0:
    # --- 1. PLOT DELLA DISTRIBUZIONE DELL'ETA --- 
    all_ages = [sample['true_age'] for sample in full_train_dataset.samples]
    
    plt.figure(figsize=(10, 5))
    plt.hist(all_ages, bins=13, range=(0, 65), edgecolor='black', color='skyblue', alpha=0.8)
    plt.title("Distribuzione dell'Età nel Dataset (Valore Medio per Fascia)", fontsize=14)
    plt.xlabel("Età (Anni)", fontsize=12)
    plt.ylabel("Numero di Pazienti", fontsize=12)
    plt.xticks(np.unique(all_ages))
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.show()
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Utilizzo dispositivo: {device}\n")
    
    # --- 2. SEPARAZIONE 10% TEST SET (STRATIFICATA) ---
    all_indices = np.arange(dataset_size)
    train_val_idx, test_idx = train_test_split(
        all_indices, 
        test_size=0.10, 
        random_state=42, 
        stratify=all_ages
    )
    
    test_dataset = torch.utils.data.Subset(full_val_dataset, test_idx)
    test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, num_workers=2)
    
    print(f"Dimensione Train/Val Set (Cross-Validation): {len(train_val_idx)} pazienti")
    print(f"Dimensione Test Set Incontaminato: {len(test_idx)} pazienti\n")
    
    # --- 3. 5-FOLD CROSS VALIDATION SUL 90% RESTANTE (STRATIFICATA) ---
    k_folds = 5
    skf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=42)
    
    fold_results = []
    best_overall_mae = float('inf')
    best_overall_model_state = None
    
    train_val_ages = np.array(all_ages)[train_val_idx]
    
    for fold, (fold_train_idx, fold_val_idx) in enumerate(skf.split(train_val_idx, train_val_ages)):
        print(f"\n==============================================")
        print(f"               FOLD {fold + 1}/{k_folds}")
        print(f"==============================================")
        
        train_idx = train_val_idx[fold_train_idx]
        val_idx = train_val_idx[fold_val_idx]
        
        train_dataset = torch.utils.data.Subset(full_train_dataset, train_idx)
        val_dataset = torch.utils.data.Subset(full_val_dataset, val_idx)
        
        train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=2)
        val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=2)
        
        model = SFCN(output_dim=70)
        if torch.cuda.device_count() > 1:
            model = nn.DataParallel(model)
        model = model.to(device)
        
        optimizer = torch.optim.SGD(model.parameters(), lr=0.01, weight_decay=0.001)
        
        trained_model, train_losses, val_losses, val_maes = train_model(
            model=model, 
            train_loader=train_loader, 
            val_loader=val_loader, 
            optimizer=optimizer, 
            device=device, 
            epochs=500,
            step_size=200,
            gamma=0.3,
            patience=100
        )
        
        best_fold_mae = min(val_maes)
        fold_results.append(best_fold_mae)
        print(f"\n-> Fine Fold {fold + 1}. Miglior MAE Validazione: {best_fold_mae:.2f} anni")
        
        if best_fold_mae < best_overall_mae:
            best_overall_mae = best_fold_mae
            best_overall_model_state = trained_model.state_dict().copy()
        
    print(f"\n==============================================")
    print(f"      RISULTATI 5-FOLD CROSS VALIDATION (STRATIFICATA)")
    print(f"==============================================")
    for idx, res in enumerate(fold_results):
        print(f"Fold {idx+1}: {res:.2f} anni")
    print(f"MAE Medio Validazione: {np.mean(fold_results):.2f} ± {np.std(fold_results):.2f} anni")
    
    # --- 4. VALUTAZIONE SUL TEST SET FINALE ---
    print(f"\n==============================================")
    print(f"      VALUTAZIONE SUL TEST SET (10%)")
    print(f"==============================================")
    
    final_model = SFCN(output_dim=70)
    if torch.cuda.device_count() > 1:
        final_model = nn.DataParallel(final_model)
    final_model.to(device)
    final_model.load_state_dict(best_overall_model_state)
    final_model.eval()
    
    bin_centers = np.arange(0, 70, 1)
    
    # Liste per salvare i risultati di tutti i pazienti del Test Set
    all_test_true = []
    all_test_pred = []
    
    with torch.no_grad():
        for inputs, labels_vect, true_age in test_loader:
            inputs = inputs.to(device)
            true_age = true_age.numpy()
            
            outputs = final_model(inputs)[0]
            outputs = outputs.view(outputs.size(0), -1)
            
            prob = torch.exp(outputs).cpu().numpy()
            predicted_age = prob @ bin_centers
            
            all_test_true.extend(true_age.tolist())
            all_test_pred.extend(predicted_age.tolist())
            
    all_test_true = np.array(all_test_true)
    all_test_pred = np.array(all_test_pred)
    errors = np.abs(all_test_pred - all_test_true)
    test_mae = np.mean(errors)
    
    print(f"\n>>> MAE SUL TEST SET CATTIVO (Incontaminato): {test_mae:.2f} anni <<<")
    
    # --- 5. PLOT FINALE DEGLI ERRORI SUL TEST SET ---
    plt.figure(figsize=(14, 6))
    
    # Subplot 1: Età Reale vs Predetta (Scatter)
    plt.subplot(1, 2, 1)
    plt.scatter(all_test_true, all_test_pred, color='dodgerblue', edgecolor='k', s=80, alpha=0.8, zorder=3)
    
    # Linea ideale y=x (dove i punti cadrebbero se il modello non sbagliasse mai)
    min_val = min(min(all_test_true), min(all_test_pred)) - 2
    max_val = max(max(all_test_true), max(all_test_pred)) + 2
    plt.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Predizione Perfetta', zorder=2)
    
    plt.title('Età Reale vs Età Predetta (Test Set)', fontsize=14)
    plt.xlabel('Età Reale (Anni)', fontsize=12)
    plt.ylabel('Età Predetta (Anni)', fontsize=12)
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6, zorder=1)
    
    # Subplot 2: Errore Assoluto a Barre per ogni Paziente
    plt.subplot(1, 2, 2)
    plt.bar(range(1, len(errors) + 1), errors, color='tomato', edgecolor='k', alpha=0.8, zorder=3)
    plt.axhline(y=test_mae, color='k', linestyle='dashed', linewidth=2, label=f'MAE Medio: {test_mae:.2f} anni', zorder=4)
    
    plt.title('Errore Assoluto per Singolo Paziente (Test Set)', fontsize=14)
    plt.xlabel('Indice Paziente', fontsize=12)
    plt.ylabel('Errore Assoluto (Anni Sbagliati)', fontsize=12)
    plt.xticks(range(1, len(errors) + 1))
    plt.legend()
    plt.grid(axis='y', linestyle='--', alpha=0.6, zorder=1)
    
    plt.tight_layout()
    plt.show()
    
    # --- 6. SALVATAGGIO DEL MODELLO FINALE ---
    model_save_path = "/kaggle/working/sfcn_best_model_fold_CV.pth"
    if isinstance(final_model, nn.DataParallel):
        torch.save(final_model.module.state_dict(), model_save_path)
    else:
        torch.save(final_model.state_dict(), model_save_path)
        
    print(f"\n[!] Pesi del modello salvati con successo in: {model_save_path}")
